In [175]:
from PySide2.QtCore import QTimer, QTime, Signal
import time
import numpy as np
import os
import sys

# Get the absolute path of the current script
script_path = os.path.abspath(r"C:\Users\YY3\GIT\squdi\src\qudi\jupyternotebooks\hom")
# Get the directory name of the script path
script_dir = os.path.dirname(script_path)
# Change the working directory to the script's directory
os.chdir(script_dir)
from hom import auto2
from hom.auto2 import *
from hom.tools import *
%gui qt

In [176]:
folder_save = r'Z:\Vlad\SnV\TPI\Electrodes_e4\178-e4-stats\TEST'
params = {
    'ple_gui': ple_gui,
    'laser_scanner_logic': laser_scanner_logic,
    'scanner_gui' : scanner_gui,
    'scanning_data_logic' : scanning_data_logic,
    'pulsestreamer' : pulsestreamer,
    'timetaggerlogic': timetaggerlogic,
    'timetagger': timetagger,
    'timetagger_remote': timetagger_remote,
    'timetaggerlogic_remote': timetaggerlogic_remote,
    'poi_manager_logic_remote': poi_manager_logic_remote,
    'poi_manager_logic': poi_manager_logic,
    'switchlogic': switchlogic,
    'ibeam_smart_remote': ibeam_smart_remote,
    'powercontroller_logic': powercontroller_logic,
    'integrate_for_mins': integrate_for_mins,
    'current_cryo': current_cryo,
    'non_active_cryo': non_active_cryo,
    'folder_save': folder_save,
    'values': values,
}

hom = auto2.StarkHOM(ao_electrodes_remote, **params)  
hom.ao_electrodes._current_channel='ao3'
hom.ao_electrodes._constraints._channel_limits = {'ao3': [-1.38, 1.38]}
hom.min_position = 80
hom.max_position = 180
hom.max_power = 30e3
hom.perpendicular_position = 0
hom.parallel_position = 21
hom.polarization_is_parallel = False

In [193]:
def start_measurement(folder):
    global iteration
    
    hom.save_tagger_plots(tag=f'iter_{iteration}', folder_path=folder)

    hom.toggle_tagger_counter_plot(False)
    hom.toggle_tagger_corr_plot(False)


    hom.toggle_tagger_counter_plot(True)

    hom.stop_dump()

    hom.toggle_tagger_counter_plot(True)
    hom.toggle_tagger_corr_plot(True)

    hom.polarization_is_parallel = True

    hom.start_dump(folder, 
                f'dump_at_iter_{iteration}'.replace('.', '_'))

In [194]:
def refocus_and_realign():
    global iteration, setpoint_story, current_measurement
    
    hom.save_tagger_plots(f'iter_{iteration}', 
                                            folder_path=os.path.join(hom.folder_save, 
                                            current_measurement))
    if iteration % 3 == 0:
            hom.measurement_mode('Off-res')
            measurement_e_hom.refocus(optimize_both=True) 
            time.sleep(1)
    
    if current_measurement == 'hom':

        if iteration % 4 == 0:
            target_laser = hom.laser_scanner_logic.scanner_target

            hom.measurement_mode('PLE')

            hom.do_ple_scan(lines=5, 
                                in_range = hom.laser_scanner_logic.scan_ranges["a"])

            hom.laser_scanner_logic.set_target_position(
                        {key: float(value) for key, value in target_laser.items()}
                    )
            
            hom.save_ple(tag=f'iter_{iteration}', 
                                            folder_name=os.path.join(hom.folder_save, 
                                                            current_measurement))

            hom.ple_gui.toggle_optimize(True)
            time.sleep(0.5)
            while hom.laser_scanner_logic.module_state() == 'locked':
                time.sleep(1)
        
        if iteration % 1 == 0:
            hom.measurement_mode('PLE')
            time.sleep(0.5)
            hom.set_green_power('atto3', 5e3)
            hom.align_resonances(dv=0.4, steps=50)
            time.sleep(0.5)
            setpoint_story.append(hom.ao_electrodes.setpoint)

    hom.measurement_mode('Off-res')
    time.sleep(0.2)
    

    if current_measurement == 'hom_detuned':
        if iteration == 0:
            hom.ao_electrodes.setpoint = -0.05
            start_measurement('hom_detuned')
    if current_measurement == 'g2_atto3':
        if iteration == 0:
            start_measurement('g2_atto3')
    
        hom.set_green_power('atto3', hom.max_power)
        hom.set_green_power('bf', 0)
    
    if current_measurement == 'g2_bf':
        if iteration == 0:
            start_measurement('g2_bf')
    
        hom.set_green_power('bf', hom.max_power)
        hom.set_green_power('atto3', 0)
        
    hom.polarization_is_parallel = True
    iteration += 1
    # setpoint_story.append(hom.ao_electrodes.setpoint)


In [195]:
def next_measurement():
    global measurements, current_measurement, query_time
    timer.stop()
    next_measurement_timer.stop()
    
    current_measurement = measurements.pop()
    iteration = 0
    timer.start(query_time * 60 * 1000)
    

    if len(measurements) < 1:
        timer.stop()
        next_measurement_timer.stop()
        return 
    
    next_measurement_timer.start(next_time)

timer = QTimer()
next_measurement_timer = QTimer()

query_time = 5 # min
timer.setInterval(query_time * 60 * 1000)
timer.timeout.connect(refocus_and_realign)

measurements = ['hom', 'hom_detuned', 'g2_bf', 'g2_atto3'][::-1]
next_time = 3 * 60 * 60 * 1000 # N hours in milliseconds
next_measurement_timer.setInterval(next_time)
next_measurement_timer.setSingleShot(True)  # Ensure the timer only fires once
next_measurement_timer.timeout.connect(next_measurement)

True

In [201]:
hom.polarization_is_parallel = True
hom.measurement_mode('Off-res')

In [202]:
iteration = 0
setpoint_story = []
start_measurement('hom')
timer.start(query_time * 60 * 1000)
current_measurement = measurements.pop()
next_measurement_timer.start(next_time)

TypeError: start_measurement() takes 1 positional argument but 2 were given

In [198]:
current_measurement='test'